# Silver data cleaning: open charge FI

In [98]:
import pandas as pd
from pathlib import Path
import openpyxl
import geopandas as gpd
import shapely
import pyogrio

print(f"openpyxl version: {openpyxl.__version__}")
print(f"gpd version: {gpd.__version__}")
print(f"shapely version: {shapely.__version__}")
print(f"pyogrio version: {pyogrio.__version__}")

print(f"Current directory: {Path.cwd()}")

openpyxl version: 3.1.5
gpd version: 1.1.4
shapely version: 2.1.2
pyogrio version: 0.13.0
Current directory: c:\DE_Course\de-week8-mini-project-team1\Silver


## Bronze -> Silver transformation

In [99]:
# Read the Bronze-level data, the CSV file into a DataFrame
bronze_df = pd.read_csv("../Bronze/open_charge_raw_FI.csv", encoding="utf-8")

# Inspect the DataFrame
print(bronze_df.shape, "\n")
print(bronze_df.columns, "\n")
print(bronze_df.dtypes)

(4966, 86) 

Index(['UserComments', 'PercentageSimilarity', 'MediaItems',
       'IsRecentlyVerified', 'DateLastVerified', 'ID', 'UUID',
       'ParentChargePointID', 'DataProviderID', 'DataProvidersReference',
       'OperatorID', 'OperatorsReference', 'UsageTypeID', 'UsageCost',
       'Connections', 'NumberOfPoints', 'GeneralComments', 'DatePlanned',
       'DateLastConfirmed', 'StatusTypeID', 'DateLastStatusUpdate',
       'MetadataValues', 'DataQualityLevel', 'DateCreated',
       'SubmissionStatusTypeID', 'DataProvider.WebsiteURL',
       'DataProvider.Comments',
       'DataProvider.DataProviderStatusType.IsProviderEnabled',
       'DataProvider.DataProviderStatusType.ID',
       'DataProvider.DataProviderStatusType.Title',
       'DataProvider.IsRestrictedEdit', 'DataProvider.IsOpenDataLicensed',
       'DataProvider.IsApprovedImport', 'DataProvider.License',
       'DataProvider.DateLastImported', 'DataProvider.ID',
       'DataProvider.Title', 'OperatorInfo.WebsiteURL',
     

In [101]:
# Define the list of columns to keep and mapping, because we want to rename some columns
column_mapping = {
    "ID": "id",
    "NumberOfPoints": "number_of_points",
    "DateCreated": "date_created", # this one needs to be split later, only one key can exist in a Python dict
    "StatusType.IsOperational": "is_operational",
    "AddressInfo.Town": "locality", # original API value, this is not a municipality in many cases
    # "AddressInfo.StateOrProvince": "region", # contains 826 invalid regions -> not usable
    "AddressInfo.Country.Title": "country",
    "AddressInfo.Latitude": "latitude",
    "AddressInfo.Longitude": "longitude",
    "fetch_timestamp": "fetch_timestamp"
}

# Select and rename
silver_df = (
    bronze_df[list(column_mapping.keys())]
    .rename(columns=column_mapping)
)

# Convert types
silver_df["date_created"] = pd.to_datetime(silver_df["date_created"])

# Derive columns
silver_df = silver_df.assign(
    year=silver_df["date_created"].dt.year,
    month=silver_df["date_created"].dt.month,
)

# Define the final column order explicitly
silver_df = silver_df[
    [
        "id",
        "number_of_points",
        "year",
        "month",
        "date_created",
        "is_operational",
        "locality", # will be converted to municipality later
        "country",
        "latitude",
        "longitude",
        "fetch_timestamp",
    ]
]

# Inspect the silver DataFrame
print(silver_df.shape, "\n")
print(silver_df.columns, "\n")
print(silver_df.dtypes)

(4966, 11) 

Index(['id', 'number_of_points', 'year', 'month', 'date_created',
       'is_operational', 'locality', 'country', 'latitude', 'longitude',
       'fetch_timestamp'],
      dtype='str') 

id                                int64
number_of_points                float64
year                              int32
month                             int32
date_created        datetime64[us, UTC]
is_operational                   object
locality                            str
country                             str
latitude                        float64
longitude                       float64
fetch_timestamp                     str
dtype: object


In [103]:
# Convert the data types to use the nullable Pandas dtypes (Int64, boolean, string) for Silver-layer data
# because they preserve missing values (<NA>)

silver_df["number_of_points"] = silver_df["number_of_points"].astype("Int64")

silver_df["year"] = silver_df["year"].astype("Int64")
silver_df["month"] = silver_df["month"].astype("Int64")

# print(silver_df["is_operational"].unique()) # [True, <NA>, False]
silver_df["is_operational"] = silver_df["is_operational"].astype("boolean")

# use string of Pandas instead of str
string_columns = ["locality", "country", "fetch_timestamp"]
silver_df[string_columns] = silver_df[string_columns].astype("string")

# use datetime for the timestamp of fetching
silver_df["fetch_timestamp"] = pd.to_datetime(silver_df["fetch_timestamp"], errors="coerce", utc=True)

# Summary of data types:
# Int64 for nullable integers
# boolean for nullable booleans
# string for text
# float64 for coordinates
# datetime for date_created and fetch_timestamp

print(silver_df.dtypes)

id                                int64
number_of_points                  Int64
year                              Int64
month                             Int64
date_created        datetime64[us, UTC]
is_operational                  boolean
locality                         string
country                          string
latitude                        float64
longitude                       float64
fetch_timestamp     datetime64[us, UTC]
dtype: object


## Silver quality checks

In [105]:
# Quality checks

# An approximate bounding box for mainland Finland (including Åland and Lapland)
# Latitude: 59.5° to 70.5°
# Longitude: 19.0° to 32.0°
FINLAND_LAT_MIN = 59.5
FINLAND_LAT_MAX = 70.5
FINLAND_LON_MIN = 19.0
FINLAND_LON_MAX = 32.0

quality_issues = []
total_rows = len(silver_df)

# Helper function for adding failed checks
def add_quality_issue(check, details, **extra_fields):
    quality_issues.append({
        "check": check,
        "status": "FAILED",
        "details": details,
        "total_rows": total_rows,
        **extra_fields
    })


# 1. Row count check
if len(silver_df) != len(bronze_df):
    add_quality_issue(
        check="ROW_COUNT",
        details=f"Bronze rows: {len(bronze_df)}, Silver rows: {len(silver_df)}",
        bronze_rows=len(bronze_df),
        silver_rows=len(silver_df)
    )


# 2. Duplicate check by id
duplicate_ids = silver_df[silver_df.duplicated(subset=["id"], keep=False)]

if not duplicate_ids.empty:
    add_quality_issue(
        check="DUPLICATE_IDS",
        details=f"Duplicate IDs found: {duplicate_ids['id'].nunique()}",
        affected_rows=len(duplicate_ids),
        duplicate_id_count=duplicate_ids["id"].nunique()
    )


# 3. Date range check - NOT USED, we want to keep all charging stations, even the ones added before 2016
# invalid_dates = silver_df[
#     silver_df["date_created"].dt.year.lt(2016)
#     | silver_df["date_created"].dt.year.gt(2026)
# ]

# if not invalid_dates.empty:
#     add_quality_issue(
#         check="DATE_RANGE",
#         details=f"Rows outside 2016-2026: {len(invalid_dates)}",
#         affected_rows=len(invalid_dates),
#         invalid_pct=round(len(invalid_dates) / total_rows * 100, 2)
#     )


# 4. Not-null checks
required_columns = [
    "id",
    "number_of_points",
    "date_created",
    "year",
    "month",
    "is_operational",
    "locality",
    "country",
    "latitude",
    "longitude",
    "fetch_timestamp",
]

for col in required_columns:
    missing_count = silver_df[col].isna().sum()

    if missing_count > 0:
        add_quality_issue(
            check="NOT_NULL",
            details=f"{col}: {missing_count} missing values",
            column=col,
            missing_values=missing_count,
            non_null_values=silver_df[col].count(),
            missing_pct=round(missing_count / total_rows * 100, 2)
        )


# 5. Invalid numeric values: number_of_points
invalid_number_of_points = silver_df[
    silver_df["number_of_points"].lt(0)
]

if not invalid_number_of_points.empty:
    add_quality_issue(
        check="INVALID_COORDINATES",
        details=f"Invalid coordinate rows: {len(invalid_coordinates)}",
        affected_rows=len(invalid_coordinates),
        invalid_pct=round(len(invalid_coordinates) / total_rows * 100, 2)
    )


# 6. Invalid numeric values: coordinates
invalid_coordinates = silver_df[
    silver_df["latitude"].lt(FINLAND_LAT_MIN)
    | silver_df["latitude"].gt(FINLAND_LAT_MAX)
    | silver_df["longitude"].lt(FINLAND_LON_MIN)
    | silver_df["longitude"].gt(FINLAND_LON_MAX)
]

if not invalid_coordinates.empty:
    add_quality_issue(
        check="INVALID_COORDINATES",
        details=f"Invalid coordinate rows: {len(invalid_coordinates)}",
        affected_rows=len(invalid_coordinates)
    )

# 7. Invalid municipality

# Read the municipalities from the official Excel
# See https://stat.fi/en/luokitukset/tupa -> "Municipalities and Regional Divisions Based on Municipalities 2026"
municipalities_and_regions_df = pd.read_excel(
    "en26_Municipalities_and_Regional_Divisions_Based_on_Municipalities_2026_in_Finnish,_Swedish_and_English.xlsx",
    usecols="A,H",
    skiprows=2,
    header=None,
    names=["municipality", "region"]
)

display(municipalities_and_regions_df.head())
print(municipalities_and_regions_df.shape)

valid_municipalities = set(
    municipalities_and_regions_df["municipality"]
    .dropna()
    .astype("string")
    .str.strip()
    .str.casefold()
)

valid_regions = set(
    municipalities_and_regions_df["region"]
    .dropna()
    .astype("string")
    .str.strip()
    .str.casefold()
)

invalid_municipalities = silver_df[
    silver_df["locality"].notna()
    & ~silver_df["locality"]
        .str.strip()
        .str.casefold()
        .isin(valid_municipalities)
].copy()

if not invalid_municipalities.empty:
    add_quality_issue(
        check="INVALID_MUNICIPALITY",
        details=f"Invalid municipality rows: {len(invalid_municipalities)}",
        affected_rows=len(invalid_municipalities),
        invalid_pct=round(len(invalid_municipalities) / total_rows * 100, 2)
    )


# 8. Invalid region
# invalid_regions = silver_df[
#     silver_df["region"].notna()
#     & ~silver_df["region"]
#         .str.strip()
#         .str.casefold()
#         .isin(valid_regions)
# ].copy()

# if not invalid_regions.empty:
#     add_quality_issue(
#         check="INVALID_REGION",
#         details=f"Invalid region rows: {len(invalid_regions)}",
#         affected_rows=len(invalid_regions),
#         invalid_pct=round(len(invalid_regions) / total_rows * 100, 2)
#     )


# Final quality report
quality_report = pd.DataFrame(quality_issues).fillna("") # replace NaN with blank cells so that it is more human-readable

if quality_report.empty:
    print("All quality checks passed.")
else:
    display(quality_report)

,municipality,region
0,Alajärvi,South Ostrobothnia
1,Alavieska,North Ostrobothnia
2,Alavus,South Ostrobothnia
3,Asikkala,Päijät-Häme
4,Askola,Uusimaa


(308, 2)


,check,status,details,total_rows,column,missing_values,non_null_values,missing_pct,affected_rows,invalid_pct
0,NOT_NULL,FAILED,number_of_points: 3066 missing values,4966,number_of_points,3066.0,1900.0,61.74,,
1,NOT_NULL,FAILED,is_operational: 1280 missing values,4966,is_operational,1280.0,3686.0,25.78,,
2,NOT_NULL,FAILED,locality: 12 missing values,4966,locality,12.0,4954.0,0.24,,
3,INVALID_COORDINATES,FAILED,Invalid coordinate rows: 5,4966,,,,,5.0,
4,INVALID_MUNICIPALITY,FAILED,Invalid municipality rows: 1547,4966,,,,,1547.0,31.15


### Do a spatial join with the official municipality polygons (locality -> municipality)

In [106]:
# Read the GeoPackage
municipalities_gdf = gpd.read_file(
    "SuomenHallinnollisetKuntajakopohjaisetAluejaot_2026_10k.gpkg"
)

print(municipalities_gdf.shape)
display(municipalities_gdf.head())

(308, 9)


c:\DE_Course\de-week8-mini-project-team1\.venv\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'SuomenHallinnollisetKuntajakopohjaisetAluejaot_2026_10k.gpkg': 'Kunta' (default), 'Maakunta', 'Elinvoimakeskus', 'Valtakunta', 'Hyvinvointialue'. Specify layer parameter to avoid this warning.
  result = read_func(


,gml_id,natcode,namefin,nameswe,landarea,freshwarea,seawarea,totalarea,geometry
0,28026789,630,Pyhäntä,Pyhäntä,810.179993,36.570000,NaN,846.750000,"MULTIPOLYGON (((458191.217 7105060.046, 458370..."
1,28025702,309,Outokumpu,Outokumpu,446.250000,137.809998,NaN,584.059998,"MULTIPOLYGON (((617807.354 6956270.87, 617128...."
2,28020116,684,Rauma,Raumo,496.309998,13.890000,599.919983,1110.119995,"MULTIPOLYGON (((211671.977 6774130.131, 211282..."
3,28022745,005,Alajärvi,Alajärvi,1008.799988,48.029999,NaN,1056.829956,"MULTIPOLYGON (((333911.129 6968504.391, 333550..."
4,28027936,614,Posio,Posio,3039.419922,505.230011,NaN,3544.649902,"MULTIPOLYGON (((538958.119 7370322.624, 539785..."


In [107]:
# Inspect the available layers
layers = gpd.list_layers(
    "SuomenHallinnollisetKuntajakopohjaisetAluejaot_2026_10k.gpkg"
)

display(layers)

,name,geometry_type
0,Kunta,Unknown
1,Maakunta,Unknown
2,Elinvoimakeskus,Unknown
3,Valtakunta,Unknown
4,Hyvinvointialue,Unknown


In [108]:
# Read only the municipality layer
municipalities_gdf = gpd.read_file(
    "SuomenHallinnollisetKuntajakopohjaisetAluejaot_2026_10k.gpkg",
    layer="kunta"
)

# display(municipalities_gdf.head())
municipalities_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 308 entries, 0 to 307
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   gml_id      308 non-null    int64   
 1   natcode     308 non-null    str     
 2   namefin     308 non-null    str     
 3   nameswe     308 non-null    str     
 4   landarea    308 non-null    float32 
 5   freshwarea  308 non-null    float32 
 6   seawarea    77 non-null     float32 
 7   totalarea   308 non-null    float32 
 8   geometry    308 non-null    geometry
dtypes: float32(4), geometry(1), int64(1), str(3)
memory usage: 17.0 KB


In [109]:
# Inspect the columns
print(municipalities_gdf.columns.tolist())

['gml_id', 'natcode', 'namefin', 'nameswe', 'landarea', 'freshwarea', 'seawarea', 'totalarea', 'geometry']


In [110]:
# Convert charging stations to points
municipalities_gdf = municipalities_gdf[
    ["natcode", "namefin", "nameswe", "geometry"]
].rename(columns={
    "natcode": "municipality_code",
    "namefin": "municipality_fi",
    "nameswe": "municipality_sv"
})

stations_gdf = gpd.GeoDataFrame(
    silver_df,
    geometry=gpd.points_from_xy(
        silver_df["longitude"],
        silver_df["latitude"]
    ),
    crs="EPSG:4326"
)

# make sure both GeoDataFrames use the same CRS
stations_gdf = stations_gdf.to_crs(municipalities_gdf.crs)

In [111]:
# Spatial join
stations_with_municipality = gpd.sjoin(
    stations_gdf,
    municipalities_gdf,
    how="left",
    predicate="within"
)

In [112]:
# Inspect the result
stations_with_municipality[
    [
        "id",
        "locality",
        "municipality_code",
        "municipality_fi",
        "municipality_sv",
        "latitude",
        "longitude"
    ]
].tail(20)

,id,locality,municipality_code,municipality_fi,municipality_sv,latitude,longitude
4946,24348,SaloVars,734,Salo,Salo,60.418764,23.164466
4947,24347,Raisio,680,Raisio,Reso,60.506355,22.159411
4948,24346,Paimio,577,Paimio,Pemar,60.443490,22.605210
4949,24345,Loviisa,434,Loviisa,Lovisa,60.485006,25.892513
4950,24343,Kouvola,286,Kouvola,Kouvola,60.858750,26.567395
4951,24341,Kotka,285,Kotka,Kotka,60.542101,26.976071
4952,24340,Imatra,153,Imatra,Imatra,61.186438,28.739853
4953,24339,Aura,019,Aura,Aura,60.642132,22.551499
4954,24242,Ikaalinen,143,Ikaalinen,Ikalis,61.756190,23.060389
4955,24238,Tampere,837,Tampere,Tammerfors,61.490842,23.767333


In [ ]:
# Continue...
# silver_df["municipality"] = stations_with_municipality["municipality_fi"]
# silver_df["municipality_code"] = stations_with_municipality["municipality_code"] # TODO: drop this?

# display(silver_df.head(20))

### Use a lookup table to add the official region

In [113]:
# 1. Prepare municipality polygons
# Make municipality polygon columns ready for spatial join
municipalities_lookup_gdf = (
    municipalities_gdf[
        ["municipality_code", "municipality_fi", "geometry"]
    ]
    .rename(columns={
        "municipality_code": "lookup_municipality_code",
        "municipality_fi": "lookup_municipality"
    })
)

# 2. Create point geometries from silver_df
stations_gdf = gpd.GeoDataFrame(
    silver_df,
    geometry=gpd.points_from_xy(
        silver_df["longitude"],
        silver_df["latitude"]
    ),
    crs="EPSG:4326"
)

stations_gdf = stations_gdf.to_crs(municipalities_lookup_gdf.crs)

# 3. Spatial join: coordinates -> official municipality
stations_with_municipality = gpd.sjoin(
    stations_gdf,
    municipalities_lookup_gdf,
    how="left",
    predicate="within"
)

# Rename to the final Silver schema
silver_df = (
    pd.DataFrame(
        stations_with_municipality.drop(
            columns=["geometry", "index_right"]
        )
    )
    .rename(columns={
        "lookup_municipality": "municipality",
        "lookup_municipality_code": "municipality_code"
    })
)
# add the region from the official mapping
municipalities_and_regions_lookup = (
    municipalities_and_regions_df.rename(columns={
        "region": "lookup_region"
    })
)

# Merge
silver_df = silver_df.merge(
    municipalities_and_regions_lookup,
    on="municipality",
    how="left"
)

silver_df = silver_df.rename(columns={
    "lookup_region": "region"
})

# After this, silver_df["region"] comes from the official municipality → region mapping, not from Bronze.




In [ ]:
display(silver_df.head(20))

# silver_df[["locality", "municipality", "municipality_code", "region"]].head()

,id,number_of_points,year,month,date_created,is_operational,locality,country,latitude,longitude,fetch_timestamp,municipality_code,municipality,region
0,496617,<NA>,2026,6,2026-06-30 06:43:00+00:00,True,Muonio,Finland,67.931264,23.811257,2026-06-30 07:14:05.354549+00:00,498,Muonio,Lapland
1,496616,<NA>,2026,6,2026-06-30 06:43:00+00:00,True,Pudasjärvi,Finland,65.360582,26.986816,2026-06-30 07:14:05.354549+00:00,615,Pudasjärvi,North Ostrobothnia
2,496615,<NA>,2026,6,2026-06-30 06:43:00+00:00,True,Oulu,Finland,64.962196,25.519501,2026-06-30 07:14:05.354549+00:00,564,Oulu,North Ostrobothnia
3,496614,<NA>,2026,6,2026-06-30 06:43:00+00:00,True,Oulu,Finland,64.930373,25.377133,2026-06-30 07:14:05.354549+00:00,564,Oulu,North Ostrobothnia
4,492114,10,2026,6,2026-06-01 18:54:00+00:00,True,Kirkkonummi,Finland,60.152403,24.529720,2026-06-30 07:14:05.354549+00:00,257,Kirkkonummi,Uusimaa
5,492113,2,2026,6,2026-06-01 18:51:00+00:00,True,Kirkkonummi,Finland,60.153643,24.531691,2026-06-30 07:14:05.354549+00:00,257,Kirkkonummi,Uusimaa
6,492112,2,2026,6,2026-06-01 18:35:00+00:00,True,Masala,Finland,60.154667,24.529397,2026-06-30 07:14:05.354549+00:00,257,Kirkkonummi,Uusimaa
7,491922,2,2026,5,2026-05-31 06:51:00+00:00,True,Hämeenkyrö,Finland,61.634280,23.200234,2026-06-30 07:14:05.354549+00:00,108,Hämeenkyrö,Pirkanmaa
8,484655,<NA>,2026,4,2026-04-09 04:38:00+00:00,True,Nuorgam,Finland,70.083900,27.909700,2026-06-30 07:14:05.354549+00:00,890,Utsjoki,Lapland
9,484654,<NA>,2026,4,2026-04-09 04:38:00+00:00,True,Utsjoki,Finland,69.941747,27.210442,2026-06-30 07:14:05.354549+00:00,890,Utsjoki,Lapland


In [116]:
# check if any region failed to map
silver_df[silver_df["region"].isna()][
    ["id", "locality", "municipality", "region", "latitude", "longitude"]
]

,id,locality,municipality,region,latitude,longitude
2990,481650,Mariehamn,Maarianhamina,NaN,60.121700,19.932699
3003,481637,Maarianhamina,Maarianhamina,NaN,60.106112,19.929446
3004,481636,Maarianhamina,Maarianhamina,NaN,60.104419,19.942798
3006,481634,Maarianhamina,Maarianhamina,NaN,60.099510,19.941055
3007,481633,Mariehamn,Maarianhamina,NaN,60.099144,19.939142
3010,481630,Mariehamn,Maarianhamina,NaN,60.096777,19.941843
3013,481627,Maarianhamina,Maarianhamina,NaN,60.081628,19.940765
3097,308553,Kerava,NaN,NaN,0.000000,0.000000
3100,307060,Mariehamn,Maarianhamina,NaN,60.101254,19.940489
3101,307058,Mariehamn,Maarianhamina,NaN,60.106796,19.942853


In [ ]:
# Drop unnecessary columns
silver_df = silver_df.drop(columns=["date_created", "locality"], errors="ignore")

In [ ]:
print(stations_with_municipality.columns.tolist())

In [ ]:
print(municipalities_gdf.columns.tolist())

In [ ]:
# If you already had a Bronze region column, remove it before the merge
silver_df = silver_df.drop(columns=["region"], errors="ignore")

In [ ]:
# Inspect the failed data in more detail

# invalid_municipalities
# invalid_municipalities.head(20)




# failing_municipality_values = (
#     invalid_municipalities["municipality"]
#     .value_counts()
#     .sort_values(ascending=False)
# )
# display(failing_municipality_values)
# failing_municipality_values.to_csv("failing_municipality_values.csv", encoding="utf-8", index=False)



invalid_municipalities[
    ["locality", "region", "latitude", "longitude"]
].sample(30, random_state=42)




# dump to a CSV file for debugging
# municipalities_df.to_csv("municipalities.csv", encoding="utf-8", index=False)
# invalid_municipalities.to_csv("invalid_municipalities.csv", encoding="utf-8", index=False)

# invalid_dates
# display(invalid_coordinates)

In [ ]:
municipalities_and_regions_df.head()